# Notebook for the 1000 Runs Ensemble

In [1]:
import pandas as pd
import os
from utils.eda_utils import EDAUtils
import boto3

In [ ]:
%load_ext autoreload
%autoreload 2

In [2]:
SCRIPT_DIR_PATH = os.getcwd()
ROOT_DIR_PATH = os.path.dirname(SCRIPT_DIR_PATH)
DATA_DIR_PATH = os.path.join(ROOT_DIR_PATH, "data")
MAPPING_DIR_PATH = os.path.join(DATA_DIR_PATH, "mapping")
SSP_DIR_PATH = os.path.join(DATA_DIR_PATH, "ssp")
TRAINING_DIR_PATH = os.path.join(DATA_DIR_PATH, "training")
CONFIG_DIR_PATH = os.path.join(ROOT_DIR_PATH, "config")

In [ ]:
os.makedirs(SSP_DIR_PATH, exist_ok=True)

In [3]:
edau = EDAUtils()

## Pull data from AWS S3


In [ ]:
aws_config = edau.read_yaml(os.path.join(CONFIG_DIR_PATH, "aws_credentials_config.yaml"))
profile_name = aws_config["profile_name"]
bucket_name = aws_config["bucket_name"]
# Set your profile
session = boto3.Session(profile_name=profile_name)

# Create an S3 client or resource
s3 = session.resource('s3')

run_id = "sisepuede_run_2025-08-28t15;29;22.344855"

# Define folder prefix
prefix = f'transfers/{run_id}/'  # this is like the "folder" in S3

In [ ]:
# Local destination
destination = os.path.join(SSP_DIR_PATH, run_id)
if os.path.exists(destination) and os.listdir(destination):
    print(f"Destination '{destination}' already exists and is not empty. Skipping download.")
else:
    os.makedirs(destination, exist_ok=True)
    bucket = s3.Bucket(bucket_name)
    for obj in bucket.objects.filter(Prefix=prefix):
        if obj.key.endswith('/') or "transformations" in obj.key:  # skip directories and transformations
            continue
        target_path = os.path.join(destination, os.path.basename(obj.key))
        print(f"Downloading {obj.key} to {target_path}")
        bucket.download_file(obj.key, target_path)

In [4]:
run_id="sisepuede_run_2025-10-14t00;19;25.886881"

In [5]:
SIMULATION_DIR_PATH = os.path.join(SSP_DIR_PATH, run_id)
print(SIMULATION_DIR_PATH)

e:\Current_2023\WI\work\ssp_louisiana\metamodel\data\ssp\sisepuede_run_2025-10-14t00;19;25.886881


## Load and Process LHC Samples Dataframes

In [6]:
# Load lhc samples dfs
lhs_exogenous_df = pd.read_csv(os.path.join(SIMULATION_DIR_PATH, "ATTRIBUTE_LHC_SAMPLES_EXOGENOUS_UNCERTAINTIES.csv"))
lhs_levers_df = pd.read_csv(os.path.join(SIMULATION_DIR_PATH, "ATTRIBUTE_LHC_SAMPLES_LEVER_EFFECTS.csv"))

In [7]:
# Check design ids
lhs_exogenous_df.head()

,region,design_id,future_id,46,47,48,49,50,51,52,53,54,55,56,57,58,59
0,louisiana,-1,1,0.362295,0.639448,0.536505,0.069870,0.972121,0.350096,0.013272,0.138693,0.698405,0.403883,0.357435,0.419629,0.323321,0.466898
1,louisiana,-1,2,0.453799,0.702236,0.097084,0.604824,0.155687,0.670921,0.492146,0.863642,0.650856,0.586198,0.895515,0.383658,0.229789,0.188339
2,louisiana,-1,3,0.120669,0.576491,0.067843,0.362784,0.809979,0.282347,0.596430,0.791199,0.560186,0.791864,0.160298,0.653489,0.195396,0.332572
3,louisiana,-1,4,0.218794,0.895455,0.538409,0.193003,0.514389,0.493241,0.053878,0.546383,0.067101,0.291212,0.380140,0.929795,0.664778,0.138639
4,louisiana,-1,5,0.340851,0.766183,0.456184,0.561235,0.079006,0.883358,0.541472,0.215089,0.662216,0.570226,0.621162,0.491896,0.417463,0.635064


In [8]:
lhs_exogenous_df.design_id.unique()

array([-1,  4])

In [9]:
lhs_exogenous_df[lhs_exogenous_df['design_id']== 0]

,region,design_id,future_id,46,47,48,49,50,51,52,53,54,55,56,57,58,59


In [10]:
lhs_levers_df.head()

,region,design_id,future_id,1,2,3,4,5,6,7,...,1761,1762,1767,1768,1770,1778,1780,1792,1793,1801
0,louisiana,-1,1,0.303285,0.037934,0.617066,0.097521,0.469836,0.216534,0.944742,...,0.440563,0.407390,0.239100,0.201661,0.933695,0.744614,0.345463,0.514843,0.040701,0.865033
1,louisiana,-1,2,0.377135,0.539486,0.857265,0.025950,0.012477,0.783614,0.586953,...,0.186101,0.822272,0.409591,0.084974,0.769561,0.318526,0.942912,0.829314,0.229475,0.697621
2,louisiana,-1,3,0.952401,0.228593,0.972167,0.960658,0.159089,0.039879,0.327936,...,0.542477,0.653018,0.966889,0.516578,0.754681,0.474827,0.817638,0.260621,0.728775,0.736069
3,louisiana,-1,4,0.765244,0.871007,0.325361,0.045184,0.537098,0.040606,0.761664,...,0.906312,0.667512,0.420655,0.735931,0.249362,0.562946,0.791030,0.769653,0.512995,0.822182
4,louisiana,-1,5,0.082978,0.171474,0.720600,0.619098,0.797104,0.821825,0.820461,...,0.166547,0.847204,0.379052,0.588185,0.687127,0.526062,0.511451,0.292011,0.825633,0.353985


In [11]:
lhs_levers_df.design_id.unique()

array([-1,  4])

In [12]:
# print shapes
print(lhs_exogenous_df.shape)
print(lhs_levers_df.shape)

(2000, 17)
(2000, 68)


In [13]:
lhs_df_merged = pd.merge(lhs_exogenous_df, lhs_levers_df, on=["region", "design_id", "future_id"], how="outer", suffixes=('_X', '_L'))
lhs_df_merged.head()

,region,design_id,future_id,46,47,48,49,50,51,52,...,1761,1762,1767,1768,1770,1778,1780,1792,1793,1801
0,louisiana,-1,1,0.362295,0.639448,0.536505,0.069870,0.972121,0.350096,0.013272,...,0.440563,0.407390,0.239100,0.201661,0.933695,0.744614,0.345463,0.514843,0.040701,0.865033
1,louisiana,-1,2,0.453799,0.702236,0.097084,0.604824,0.155687,0.670921,0.492146,...,0.186101,0.822272,0.409591,0.084974,0.769561,0.318526,0.942912,0.829314,0.229475,0.697621
2,louisiana,-1,3,0.120669,0.576491,0.067843,0.362784,0.809979,0.282347,0.596430,...,0.542477,0.653018,0.966889,0.516578,0.754681,0.474827,0.817638,0.260621,0.728775,0.736069
3,louisiana,-1,4,0.218794,0.895455,0.538409,0.193003,0.514389,0.493241,0.053878,...,0.906312,0.667512,0.420655,0.735931,0.249362,0.562946,0.791030,0.769653,0.512995,0.822182
4,louisiana,-1,5,0.340851,0.766183,0.456184,0.561235,0.079006,0.883358,0.541472,...,0.166547,0.847204,0.379052,0.588185,0.687127,0.526062,0.511451,0.292011,0.825633,0.353985


In [14]:
# Filter the lhs_df_merged to only include rows where design_id is 4
lhs_df_merged = lhs_df_merged[lhs_df_merged.design_id == 4]
lhs_df_merged.shape

(1000, 82)

In [15]:
lhs_df_merged.design_id.unique()

array([4])

In [16]:
# NOTE: check col names, there should be no duplicates
lhs_df_merged.columns

Index(['region', 'design_id', 'future_id', '46', '47', '48', '49', '50', '51',
       '52', '53', '54', '55', '56', '57', '58', '59', '1', '2', '3', '4', '5',
       '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17',
       '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29',
       '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41',
       '42', '43', '44', '45', '1714', '1715', '1718', '1720', '1723', '1740',
       '1741', '1744', '1747', '1749', '1761', '1762', '1767', '1768', '1770',
       '1778', '1780', '1792', '1793', '1801'],
      dtype='object')

In [17]:
lhs_df_merged.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1000 entries, 1000 to 1999
Data columns (total 82 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   region     1000 non-null   object 
 1   design_id  1000 non-null   int64  
 2   future_id  1000 non-null   int64  
 3   46         1000 non-null   float64
 4   47         1000 non-null   float64
 5   48         1000 non-null   float64
 6   49         1000 non-null   float64
 7   50         1000 non-null   float64
 8   51         1000 non-null   float64
 9   52         1000 non-null   float64
 10  53         1000 non-null   float64
 11  54         1000 non-null   float64
 12  55         1000 non-null   float64
 13  56         1000 non-null   float64
 14  57         1000 non-null   float64
 15  58         1000 non-null   float64
 16  59         1000 non-null   float64
 17  1          1000 non-null   float64
 18  2          1000 non-null   float64
 19  3          1000 non-null   float64
 20  4         

## Load SISEPUEDE WIDE_INPUTS_OUTPUTS

In [18]:
attr_primary_df = pd.read_csv(os.path.join(SIMULATION_DIR_PATH, "ATTRIBUTE_PRIMARY.csv"))
attr_primary_df

,primary_id,design_id,strategy_id,future_id
0,368368,4,0,0
1,438438,4,6004,0
2,438439,4,6004,1
3,438440,4,6004,2
4,438441,4,6004,3
...,...,...,...,...
997,439434,4,6004,996
998,439435,4,6004,997
999,439436,4,6004,998
1000,439437,4,6004,999


In [19]:
attr_primary_df = attr_primary_df[attr_primary_df["strategy_id"].isin([6004])]
attr_primary_df

,primary_id,design_id,strategy_id,future_id
1,438438,4,6004,0
2,438439,4,6004,1
3,438440,4,6004,2
4,438441,4,6004,3
5,438442,4,6004,4
...,...,...,...,...
997,439434,4,6004,996
998,439435,4,6004,997
999,439436,4,6004,998
1000,439437,4,6004,999


In [20]:
wide_inputs_outputs_df = pd.read_csv(os.path.join(SIMULATION_DIR_PATH, "sisepuede_results_IDE_2025-10-14t00;19;25.886881_cleaned.csv"))
wide_inputs_outputs_df

,primary_id,region,time_period,area_agrc_crops_bevs_and_spices,area_agrc_crops_cereals,area_agrc_crops_fibers,area_agrc_crops_fruits,area_agrc_crops_herbs_and_other_perennial_crops,area_agrc_crops_nuts,area_agrc_crops_other_annual,...,yf_agrc_herbs_and_other_perennial_crops_tonne_ha,yf_agrc_nuts_tonne_ha,yf_agrc_other_annual_tonne_ha,yf_agrc_other_woody_perennial_tonne_ha,yf_agrc_pulses_tonne_ha,yf_agrc_rice_tonne_ha,yf_agrc_sugar_cane_tonne_ha,yf_agrc_tubers_tonne_ha,yf_agrc_vegetables_and_vines_tonne_ha,yf_lndu_supremum_pastures_tonne_per_ha
0,368368,louisiana,7,0,356696.043492,66146.621297,77.773211,76769.151304,6508.599365,1.119173e+06,...,12.023165,2.949341,6.177415,0,3.474771,8.253027,87.719298,39.935028,30.906144,92.81
1,368368,louisiana,8,0,355221.860914,65873.245131,77.451784,76451.873477,6481.700093,1.114548e+06,...,12.023165,2.949341,5.189029,0,2.688411,8.475414,83.271559,39.935028,30.906144,92.81
2,368368,louisiana,9,0,353750.075712,65600.313541,77.130879,76135.111621,6454.844566,1.109930e+06,...,12.023165,2.949341,6.319971,0,3.416092,8.422668,89.334930,39.935028,30.906144,92.81
3,368368,louisiana,10,0,352280.764654,65327.840762,76.810514,75818.882257,6428.034184,1.105320e+06,...,12.023165,2.949341,6.319971,0,3.416092,8.422668,89.334930,39.935028,30.906144,92.81
4,368368,louisiana,11,0,350814.002751,65055.840705,76.490704,75503.201530,6401.270317,1.100718e+06,...,12.023165,2.949341,6.319971,0,3.416092,8.422668,89.334930,39.935028,30.906144,92.81
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28386,439438,louisiana,31,0,394099.600570,66907.447597,96.502416,91156.969998,8536.698966,1.354004e+06,...,12.023165,2.949341,6.319971,0,3.416092,8.422668,89.334930,39.935028,30.906144,92.81
28387,439438,louisiana,32,0,397151.340963,67357.976820,97.689091,91932.200996,8646.210808,1.369353e+06,...,12.023165,2.949341,6.319971,0,3.416092,8.422668,89.334930,39.935028,30.906144,92.81
28388,439438,louisiana,33,0,400055.592031,67801.160115,98.824247,92655.158178,8749.992059,1.384002e+06,...,12.023165,2.949341,6.319971,0,3.416092,8.422668,89.334930,39.935028,30.906144,92.81
28389,439438,louisiana,34,0,402829.889014,68236.081895,99.915282,93333.892444,8848.955183,1.398046e+06,...,12.023165,2.949341,6.319971,0,3.416092,8.422668,89.334930,39.935028,30.906144,92.81


In [21]:
wide_inputs_outputs_df['primary_id'].unique()

array([368368, 438438, 438439, 438440, 438441, 438442, 438443, 438444,
       438445, 438446, 438447, 438448, 438449, 438450, 438451, 438452,
       438453, 438454, 438455, 438456, 438457, 438458, 438459, 438460,
       438461, 438462, 438463, 438464, 438465, 438466, 438467, 438468,
       438469, 438470, 438471, 438472, 438473, 438474, 438475, 438476,
       438477, 438478, 438479, 438480, 438481, 438482, 438483, 438484,
       438485, 438486, 438487, 438488, 438489, 438490, 438491, 438492,
       438493, 438494, 438495, 438496, 438497, 438498, 438499, 438500,
       438501, 438502, 438503, 438504, 438505, 438506, 438507, 438508,
       438509, 438510, 438511, 438512, 438513, 438514, 438515, 438516,
       438517, 438518, 438519, 438520, 438521, 438522, 438523, 438524,
       438525, 438526, 438527, 438528, 438529, 438530, 438531, 438532,
       438533, 438534, 438535, 438536, 438537, 438538, 438539, 438540,
       438541, 438542, 438543, 438544, 438545, 438546, 438547, 438548,
      

In [22]:
wide_inputs_outputs_df = wide_inputs_outputs_df[wide_inputs_outputs_df["primary_id"].isin(attr_primary_df["primary_id"].unique())]

In [23]:
wide_inputs_outputs_df['primary_id'].unique()

array([438438, 438439, 438440, 438441, 438442, 438443, 438444, 438445,
       438446, 438447, 438448, 438449, 438450, 438451, 438452, 438453,
       438454, 438455, 438456, 438457, 438458, 438459, 438460, 438461,
       438462, 438463, 438464, 438465, 438466, 438467, 438468, 438469,
       438470, 438471, 438472, 438473, 438474, 438475, 438476, 438477,
       438478, 438479, 438480, 438481, 438482, 438483, 438484, 438485,
       438486, 438487, 438488, 438489, 438490, 438491, 438492, 438493,
       438494, 438495, 438496, 438497, 438498, 438499, 438500, 438501,
       438502, 438503, 438504, 438505, 438506, 438507, 438508, 438509,
       438510, 438511, 438512, 438513, 438514, 438515, 438516, 438517,
       438518, 438519, 438520, 438521, 438522, 438523, 438524, 438525,
       438526, 438527, 438528, 438529, 438530, 438531, 438532, 438533,
       438534, 438535, 438536, 438537, 438538, 438539, 438540, 438541,
       438542, 438543, 438544, 438545, 438546, 438547, 438548, 438549,
      

In [24]:
wide_inputs_outputs_df.primary_id.nunique()

978

## Load Costs-Benefits Data

In [51]:
cb_df = pd.read_csv(os.path.join(SIMULATION_DIR_PATH, "wide_combined_cb_results_2025-10-14t00;19;25.886881.csv"))
cb_df


,primary_id,future_id,strategy_code,Year,air_pollution,congestion,consumer_savings,crop_value,ecosystem_services,env_pollution,...,human_health,ippu_value,land_pollution,lvst_value,road_safety,sector_specific,system_cost,technical_cost,technical_savings,water_pollution
0,438438,0,PFLO:ALL_LA_ACTIONS,2022.0,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.0,...,0.000000,0.000000,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000
1,438438,0,PFLO:ALL_LA_ACTIONS,2023.0,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.0,...,0.000000,0.000000,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000
2,438438,0,PFLO:ALL_LA_ACTIONS,2024.0,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.0,...,0.000000,0.000000,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000
3,438438,0,PFLO:ALL_LA_ACTIONS,2025.0,1.230503e-15,-6.072606e-19,0.021749,-1.083523e-13,7.945404e-12,0.0,...,0.019772,0.000000,3.001333e-16,0.000000,-7.035322e-19,3.890861e-16,0.000000e+00,-0.004080,-3.744519e-16,0.000000
4,438438,0,PFLO:ALL_LA_ACTIONS,2026.0,2.101911e-15,-6.072461e-19,0.043782,-1.210061e-13,7.081794e-12,0.0,...,0.039802,0.000000,3.001333e-16,0.000000,-7.035073e-19,5.940563e-16,-5.066395e-17,-0.005316,-3.384234e-16,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28357,439438,1000,PFLO:ALL_LA_ACTIONS,2046.0,2.385517e+00,3.440483e+00,0.641007,1.181734e+00,-9.874594e-03,0.0,...,0.458027,0.011771,-2.894399e-03,-0.011137,5.273865e+00,1.107231e+01,1.015131e+01,-7.184907,1.174754e+00,-0.174743
28358,439438,1000,PFLO:ALL_LA_ACTIONS,2047.0,2.491444e+00,3.639056e+00,0.673672,1.254661e+00,-7.482571e-03,0.0,...,0.480250,0.012772,-3.023546e-03,-0.015350,5.585800e+00,1.151296e+01,1.089828e+01,-7.513685,1.245072e+00,-0.185693
28359,439438,1000,PFLO:ALL_LA_ACTIONS,2048.0,2.597105e+00,3.842546e+00,0.706387,1.325132e+00,-4.589885e-03,0.0,...,0.502595,0.013819,-3.142271e-03,-0.019891,5.906132e+00,1.195487e+01,1.167529e+01,-7.974180,1.316646e+00,-0.196840
28360,439438,1000,PFLO:ALL_LA_ACTIONS,2049.0,2.702629e+00,4.051158e+00,0.739175,1.393433e+00,-1.239693e-03,0.0,...,0.525065,0.014912,-3.251901e-03,-0.024720,6.235185e+00,1.239778e+01,1.248331e+01,-8.275554,1.389500e+00,-0.208189


In [52]:
cb_df.future_id.nunique()

978

In [53]:
cb_df = cb_df[cb_df["primary_id"].isin(attr_primary_df["primary_id"].unique())]


In [54]:
# Keep only rows whose primary_id appears in wide_inputs_outputs_df
keep_ids = set(wide_inputs_outputs_df['primary_id'].unique())
cb_df = cb_df[cb_df['primary_id'].isin(keep_ids)].copy()

In [55]:
cb_df.primary_id.nunique()

978

## Load LSU data

In [30]:
lsu_data = pd.read_csv(os.path.join(SIMULATION_DIR_PATH, "lsu_output_1000_ensemble_10_14_updated.csv"))
lsu_data.head()

,time,primary_id,la_value_direct,la_value_indirect,la_value_induced,la_earnings_direct,la_earnings_indirect,la_earnings_induced,la_employment_direct,la_employment_indirect,la_employment_total
0,6,438438,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
1,7,438438,-8.910894e-06,-3.024936e-06,8.359551e-06,-2.190471e-06,-1.750886e-06,5.125999e-06,-2.091838e-11,-2.319211e-11,7.912604e-11
2,8,438438,-4.470348e-07,-2.235174e-07,-2.831221e-07,-3.576279e-07,-1.043081e-07,-1.564622e-07,-5.911716e-12,-1.705303e-12,-1.091394e-11
3,9,438438,-7.182360e-06,-2.160668e-06,5.394220e-06,-1.832843e-06,-1.311302e-06,3.166497e-06,-1.773515e-11,-1.830358e-11,4.547474e-11
4,10,438438,-6.258488e-07,-3.278255e-07,-4.172325e-07,-5.364418e-07,-1.639128e-07,-2.309680e-07,-8.640200e-12,-2.614797e-12,-1.637090e-11


In [31]:
lsu_data = lsu_data[lsu_data["primary_id"].isin(attr_primary_df["primary_id"].unique())]

In [32]:
lsu_data.primary_id.nunique()

978

In [33]:
lsu_data=lsu_data.rename(columns={"la_employment_total": "num_jobs"})

In [ ]:
lsu_data

In [34]:
lsu_data["num_jobs"].describe()

count     29340.000000
mean       9915.241243
std       17039.069790
min     -105771.048666
25%          -0.010985
50%        8960.940143
75%       21746.804419
max       80847.278163
Name: num_jobs, dtype: float64

In [35]:
lsu_data["earning_per_job"] = (
    lsu_data[["la_earnings_direct", "la_earnings_indirect", "la_earnings_induced"]].sum(axis=1, min_count=1)
    / lsu_data["num_jobs"].replace(0, pd.NA)
)


In [36]:
lsu_data["earning_per_job"].describe()

count     28362.000000
unique    25765.000000
top       17248.238457
freq        106.000000
Name: earning_per_job, dtype: float64

In [37]:
lsu_data.primary_id.nunique()

978

In [ ]:
lsu_data

## Data Cleaning

### SISEPUEDE Emission data

In [38]:
# Get the subsector total variables
subsector_total_vars = [c for c in wide_inputs_outputs_df.columns if "emission_co2e_subsector_total" in c]

In [39]:
# Filter to only subsector total columns and primary_id, time_period
la_emissions_df = wide_inputs_outputs_df[["primary_id", "time_period"] + subsector_total_vars]
la_emissions_df.head()

,primary_id,time_period,emission_co2e_subsector_total_agrc,emission_co2e_subsector_total_ccsq,emission_co2e_subsector_total_entc,emission_co2e_subsector_total_fgtv,emission_co2e_subsector_total_frst,emission_co2e_subsector_total_inen,emission_co2e_subsector_total_ippu,emission_co2e_subsector_total_lndu,emission_co2e_subsector_total_lsmm,emission_co2e_subsector_total_lvst,emission_co2e_subsector_total_scoe,emission_co2e_subsector_total_soil,emission_co2e_subsector_total_trns,emission_co2e_subsector_total_trww,emission_co2e_subsector_total_waso
29,438438,7,2.924249,0.0,35.431379,13.094104,-36.545088,117.042622,2.885346,-0.077001,0.924671,0.845459,4.491483,1.233181,45.130223,0.447204,3.138402
30,438438,8,2.864242,0.0,34.565234,13.165887,-36.294903,114.905242,2.893930,-0.085764,0.922202,0.832727,4.525430,1.225662,45.865290,0.454013,3.189568
31,438438,9,2.912181,0.0,33.953653,13.310116,-36.132908,116.108554,2.904645,-0.094510,0.920094,0.820192,4.561021,1.213064,46.698636,0.461144,3.239706
32,438438,10,2.900085,0.0,33.621356,13.289368,-35.699353,116.039398,2.917465,-0.103238,0.918245,0.807853,4.598409,1.194657,47.611029,0.468528,3.291336
33,438438,11,2.888010,0.0,33.798238,13.314048,-35.379533,116.143155,2.932343,-0.111948,0.916637,0.795706,4.637659,1.170317,48.587463,0.476114,3.343789


In [40]:
la_emissions_df.tail()

,primary_id,time_period,emission_co2e_subsector_total_agrc,emission_co2e_subsector_total_ccsq,emission_co2e_subsector_total_entc,emission_co2e_subsector_total_fgtv,emission_co2e_subsector_total_frst,emission_co2e_subsector_total_inen,emission_co2e_subsector_total_ippu,emission_co2e_subsector_total_lndu,emission_co2e_subsector_total_lsmm,emission_co2e_subsector_total_lvst,emission_co2e_subsector_total_scoe,emission_co2e_subsector_total_soil,emission_co2e_subsector_total_trns,emission_co2e_subsector_total_trww,emission_co2e_subsector_total_waso
28386,439438,31,3.146551,-10.962546,46.921995,16.499439,-22.084523,55.781309,2.467431,-0.828532,0.537670,0.713817,3.987120,1.815144,27.730195,0.519819,7.109860
28387,439438,32,3.164776,-11.539522,47.410256,16.048871,-21.691710,53.971814,2.422463,-0.886190,0.511884,0.697888,3.974305,1.835858,27.181664,0.522400,7.315643
28388,439438,33,3.181390,-12.116498,47.915342,15.599155,-21.323487,52.227981,2.377047,-0.942069,0.485924,0.680952,3.963841,1.866267,26.657231,0.525030,7.522671
28389,439438,34,3.196578,-12.693475,48.437348,15.150222,-20.980072,50.539069,2.331070,-0.991002,0.459920,0.663174,3.955552,1.874873,26.153741,0.527707,7.730972
28390,439438,35,3.210537,-13.270451,48.964904,14.701639,-20.875331,48.366063,2.284396,-1.001408,0.433987,0.644709,3.949323,1.876416,25.667608,0.530422,7.940522


### Production Data

In [41]:
# 2) Define your fuels and sectors
relevant_fuels = ['biomass', 
                    'coal', 
                    'coke', 
                    'diesel', 
                    'electricity',
                    'furnace_gas',
                    'gasoline', 
                    'hydrocarbon_gas_liquids',
                    'hydrogen',
                    'kerosene',
                    'natural_gas',
                    'natural_gas_liquid',
                    'oil']

sectors = ['agriculture_and_livestock',
           'cement',
           'chemicals',
           'electronics',
           'glass',
           'lime_and_carbonite',
           'metals',
           'mining',
           'other_product_manufacturing',
           'paper',
           'plastic',
           'recycled_glass',
           'recycled_metals',
           'recycled_paper',
           'recycled_plastic',
           'recycled_rubber_and_leather',
           'recycled_textiles',
           'recycled_wood',
           'rubber_and_leather',
           'textiles',
           'wood']

In [42]:
# 4) Industrial cost parameters
capex_industrial_electricity = 92666.6 * 21
capex_industrial_other       = 92666.6 * 12
opex_industrial_electricity  = 92666.6 * 2.5
opex_industrial_other        = 92666.6 * 4.5
capex_multiplier_efficiency = 10000000
opex_multiplier_efficiency = 0

In [43]:
# 5) Loop over fuels and sectors
# Use the original dataframe directly
# Initialize results dataframe
ind_fuel_demand_by_sector = pd.DataFrame({
    'primary_id': wide_inputs_outputs_df['primary_id'],
    'time_period': wide_inputs_outputs_df['time_period']
}, index=wide_inputs_outputs_df.index)

# Helper alias for tricky fuel names
alias = {
    'natural_gas_liquid': ['natural_gas_liquid', 'natural_gas_liquids'],
    'biomass': ['biomass', 'solid_biomass'],   # <-- added
}

for fuel in relevant_fuels:
    fuel_keys = alias.get(fuel, [fuel])  # use aliases everywhere for this fuel

    # ------------------------------------------------------------
    # ENTc (fuel-only) — handled OUTSIDE sector loop
    # ------------------------------------------------------------
    entc_found_col = None
    for k in fuel_keys:
        cand = f'energy_demand_enfu_subsector_total_pj_entc_fuel_{k}'
        if cand in wide_inputs_outputs_df.columns:
            entc_found_col = cand
            break
    if entc_found_col is not None:
        ind_fuel_demand_by_sector[entc_found_col] = wide_inputs_outputs_df[entc_found_col]

    # ------------------------------------------------------------
    # Efficiency column (accept any alias)
    # ------------------------------------------------------------
    eff_cols = [
        c for c in wide_inputs_outputs_df.columns
        if any(c.startswith(f'efficfactor_enfu_industrial_energy_fuel_{k}') for k in fuel_keys)
    ]
    if not eff_cols:
        continue
    fuel_efficiency = wide_inputs_outputs_df[eff_cols[0]]

    # ------------------------------------------------------------
    # Sector loop (unchanged logic; now alias-aware for fractions)
    # ------------------------------------------------------------
    for sector in sectors:
        sector_dem_cols = [c for c in wide_inputs_outputs_df.columns
                           if f'energy_demand_inen_{sector}' in c]
        sector_fuel_fraction_cols = [
            c for c in wide_inputs_outputs_df.columns
            if any(f'frac_inen_energy_{sector}_{k}' in c for k in fuel_keys)
        ]

        if sector_dem_cols and sector_fuel_fraction_cols:
            sector_total_demand = wide_inputs_outputs_df[sector_dem_cols[0]]
            sector_fuel_fraction = wide_inputs_outputs_df[sector_fuel_fraction_cols[0]]

            has_any_demand = ((sector_fuel_fraction * sector_total_demand).fillna(0) != 0).any()
            if has_any_demand or fuel == 'electricity':
                sector_fuel_demand = sector_fuel_fraction * sector_total_demand
                ind_fuel_demand_by_sector[f'energy_demand_{sector}_{fuel}'] = sector_fuel_demand

                # CAPEX/OPEX
                if fuel == 'electricity':
                    ind_fuel_demand_by_sector[f'energy_demand_capex_{sector}_{fuel}'] = (
                        sector_fuel_demand * capex_industrial_electricity
                    )
                    ind_fuel_demand_by_sector[f'energy_demand_opex_{sector}_{fuel}'] = (
                        sector_fuel_demand * opex_industrial_electricity
                    )
                else:
                    ind_fuel_demand_by_sector[f'energy_demand_capex_{sector}_{fuel}'] = (
                        sector_fuel_demand * capex_industrial_other
                    )
                    ind_fuel_demand_by_sector[f'energy_demand_opex_{sector}_{fuel}'] = (
                        sector_fuel_demand * opex_industrial_other
                    )

                # Fuel consumed + deltas
                sector_fuel_consumed = sector_fuel_demand / fuel_efficiency
                sector_fuel_consumed_baseline = (
                    sector_fuel_consumed.groupby(wide_inputs_outputs_df['primary_id']).transform('first')
                )
                sector_change_in_fuel_consumed = sector_fuel_consumed_baseline - sector_fuel_consumed
                ind_fuel_demand_by_sector[f'efficiency_energy_saving_{sector}_{fuel}'] = sector_change_in_fuel_consumed
                ind_fuel_demand_by_sector[f'efficiency_capex_{sector}_{fuel}'] = (
                    sector_change_in_fuel_consumed * capex_multiplier_efficiency
                )
                ind_fuel_demand_by_sector[f'efficiency_opex_{sector}_{fuel}'] = (
                    sector_change_in_fuel_consumed * opex_multiplier_efficiency
                )


C:\Users\nasta\AppData\Local\Temp\ipykernel_32512\2925024214.py:59: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  ind_fuel_demand_by_sector[f'energy_demand_{sector}_{fuel}'] = sector_fuel_demand
C:\Users\nasta\AppData\Local\Temp\ipykernel_32512\2925024214.py:70: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  ind_fuel_demand_by_sector[f'energy_demand_capex_{sector}_{fuel}'] = (
C:\Users\nasta\AppData\Local\Temp\ipykernel_32512\2925024214.py:73: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of call

In [ ]:
ind_fuel_demand_by_sector

In [ ]:
subset= ind_fuel_demand_by_sector.filter(regex=r'^energy_demand')


Fix natural gas passthrough

In [44]:
ind_fuel_demand_by_sector['energy_demand_enfu_subsector_total_pj_entc_fuel_natural_gas'] = ind_fuel_demand_by_sector['energy_demand_enfu_subsector_total_pj_entc_fuel_natural_gas']-wide_inputs_outputs_df['prod_enfu_fuel_natural_gas_liquid_pj']

In [45]:
ind_fuel_demand_by_sector['energy_demand_enfu_subsector_total_pj_entc_fuel_natural_gas']

29       1107.251323
30       1134.978733
31       1116.376399
32       1139.881193
33       1143.479825
            ...     
28386    1868.549474
28387    1877.098035
28388    1885.990282
28389    1895.228474
28390    1904.325934
Name: energy_demand_enfu_subsector_total_pj_entc_fuel_natural_gas, Length: 28362, dtype: float64

Add crude

In [46]:
#ind_fuel_demand_by_sector['energy_demand_enfu_subsector_total_pj_entc_fuel_crude'] = wide_inputs_outputs_df['energy_demand_enfu_subsector_total_pj_entc_fuel_crude']

C:\Users\nasta\AppData\Local\Temp\ipykernel_32512\1728094170.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  ind_fuel_demand_by_sector['energy_demand_enfu_subsector_total_pj_entc_fuel_crude'] = wide_inputs_outputs_df['energy_demand_enfu_subsector_total_pj_entc_fuel_crude']


In [47]:
#ind_fuel_demand_by_sector['energy_demand_enfu_subsector_total_pj_entc_fuel_crude']

29       10719.051847
30       10500.513468
31       10332.462853
32       10352.089213
33       10323.027379
             ...     
28386     8501.763756
28387     8441.227936
28388     8382.474026
28389     8325.449443
28390     8269.779478
Name: energy_demand_enfu_subsector_total_pj_entc_fuel_crude, Length: 28362, dtype: float64

In [48]:
ind_fuel_demand_by_sector.isna().sum().sum() 

np.int64(0)

### CB Data

In [56]:
# Make all column names lowercase
cb_df.columns = [c.lower() for c in cb_df.columns]

# Filter to only important cb columns
cb_df = cb_df[["primary_id",
               "future_id",
               "year",
               "technical_cost",
               #"consumer_savings",
               #"human_health",
               "air_pollution"]]

cb_df.head()

,primary_id,future_id,year,technical_cost,air_pollution
0,438438,0,2022.0,0.000000,0.000000e+00
1,438438,0,2023.0,0.000000,0.000000e+00
2,438438,0,2024.0,0.000000,0.000000e+00
3,438438,0,2025.0,-0.004080,1.230503e-15
4,438438,0,2026.0,-0.005316,2.101911e-15


In [57]:
cb_df.tail()

,primary_id,future_id,year,technical_cost,air_pollution
28357,439438,1000,2046.0,-7.184907,2.385517
28358,439438,1000,2047.0,-7.513685,2.491444
28359,439438,1000,2048.0,-7.974180,2.597105
28360,439438,1000,2049.0,-8.275554,2.702629
28361,439438,1000,2050.0,-8.545201,2.821515


In [58]:
cb_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28362 entries, 0 to 28361
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   primary_id      28362 non-null  int64  
 1   future_id       28362 non-null  int64  
 2   year            28362 non-null  float64
 3   technical_cost  28362 non-null  float64
 4   air_pollution   28362 non-null  float64
dtypes: float64(3), int64(2)
memory usage: 1.1 MB


### LSU data

In [59]:
lsu_data = lsu_data[["primary_id",
               "time",
               "num_jobs",
               "earning_per_job"]]

lsu_data.head()

,primary_id,time,num_jobs,earning_per_job
0,438438,6,0.000000e+00,<NA>
1,438438,7,7.912604e-11,14971.586207
2,438438,8,-1.091394e-11,56661.333333
3,438438,9,4.547474e-11,491.52
4,438438,10,-1.637090e-11,56888.888889


In [60]:
lsu_data= lsu_data.dropna()

In [61]:
lsu_data

,primary_id,time,num_jobs,earning_per_job
1,438438,7,7.912604e-11,14971.586207
2,438438,8,-1.091394e-11,56661.333333
3,438438,9,4.547474e-11,491.52
4,438438,10,-1.637090e-11,56888.888889
5,438438,11,2.680509e+01,64709.62299
...,...,...,...,...
29335,439438,31,1.518724e+04,81026.402641
29336,439438,32,1.586526e+04,81151.92643
29337,439438,33,1.589483e+04,82113.052029
29338,439438,34,1.524502e+04,83941.264701


In [62]:
lsu_data['earning_per_job'].describe()

count     28362.000000
unique    25765.000000
top       17248.238457
freq        106.000000
Name: earning_per_job, dtype: float64

## LHS Data

In [63]:
lhs_df_merged = lhs_df_merged.drop(columns=["design_id", "region"])
lhs_df_merged.head()

,future_id,46,47,48,49,50,51,52,53,54,...,1761,1762,1767,1768,1770,1778,1780,1792,1793,1801
1000,1,0.362295,0.639448,0.536505,0.069870,0.972121,0.350096,0.013272,0.138693,0.698405,...,0.440563,0.407390,0.239100,0.201661,0.933695,0.744614,0.345463,0.514843,0.040701,0.865033
1001,2,0.453799,0.702236,0.097084,0.604824,0.155687,0.670921,0.492146,0.863642,0.650856,...,0.186101,0.822272,0.409591,0.084974,0.769561,0.318526,0.942912,0.829314,0.229475,0.697621
1002,3,0.120669,0.576491,0.067843,0.362784,0.809979,0.282347,0.596430,0.791199,0.560186,...,0.542477,0.653018,0.966889,0.516578,0.754681,0.474827,0.817638,0.260621,0.728775,0.736069
1003,4,0.218794,0.895455,0.538409,0.193003,0.514389,0.493241,0.053878,0.546383,0.067101,...,0.906312,0.667512,0.420655,0.735931,0.249362,0.562946,0.791030,0.769653,0.512995,0.822182
1004,5,0.340851,0.766183,0.456184,0.561235,0.079006,0.883358,0.541472,0.215089,0.662216,...,0.166547,0.847204,0.379052,0.588185,0.687127,0.526062,0.511451,0.292011,0.825633,0.353985


## Transform time series format into single-row format

### SISEPUEDE Emission data

In [64]:
# Sum all the subsector emission columns across axis=1
la_emission_total_df = la_emissions_df.copy()
la_emission_total_df["emission_total"] = la_emission_total_df[subsector_total_vars].sum(axis=1)
la_emission_total_df.head()

,primary_id,time_period,emission_co2e_subsector_total_agrc,emission_co2e_subsector_total_ccsq,emission_co2e_subsector_total_entc,emission_co2e_subsector_total_fgtv,emission_co2e_subsector_total_frst,emission_co2e_subsector_total_inen,emission_co2e_subsector_total_ippu,emission_co2e_subsector_total_lndu,emission_co2e_subsector_total_lsmm,emission_co2e_subsector_total_lvst,emission_co2e_subsector_total_scoe,emission_co2e_subsector_total_soil,emission_co2e_subsector_total_trns,emission_co2e_subsector_total_trww,emission_co2e_subsector_total_waso,emission_total
29,438438,7,2.924249,0.0,35.431379,13.094104,-36.545088,117.042622,2.885346,-0.077001,0.924671,0.845459,4.491483,1.233181,45.130223,0.447204,3.138402,190.966232
30,438438,8,2.864242,0.0,34.565234,13.165887,-36.294903,114.905242,2.893930,-0.085764,0.922202,0.832727,4.525430,1.225662,45.865290,0.454013,3.189568,189.028760
31,438438,9,2.912181,0.0,33.953653,13.310116,-36.132908,116.108554,2.904645,-0.094510,0.920094,0.820192,4.561021,1.213064,46.698636,0.461144,3.239706,190.875588
32,438438,10,2.900085,0.0,33.621356,13.289368,-35.699353,116.039398,2.917465,-0.103238,0.918245,0.807853,4.598409,1.194657,47.611029,0.468528,3.291336,191.855135
33,438438,11,2.888010,0.0,33.798238,13.314048,-35.379533,116.143155,2.932343,-0.111948,0.916637,0.795706,4.637659,1.170317,48.587463,0.476114,3.343789,193.511999


In [65]:
# Keep only the primary_id, time_period, and emission_total columns
la_emission_total_df = la_emission_total_df[["primary_id", "time_period", "emission_total"]]
la_emission_total_df.head()

,primary_id,time_period,emission_total
29,438438,7,190.966232
30,438438,8,189.028760
31,438438,9,190.875588
32,438438,10,191.855135
33,438438,11,193.511999


In [66]:
la_emission_total_df.tail()

,primary_id,time_period,emission_total
28386,439438,31,133.354749
28387,439438,32,130.940400
28388,439438,33,128.620776
28389,439438,34,126.355679
28390,439438,35,123.423337


### Emission data sum

In [67]:
# aggregate data by primary_id summing the emissions
la_emission_df_sum_agg = la_emission_total_df.groupby(["primary_id"]).sum().reset_index()

# Drop time period column
la_emission_df_sum_agg = la_emission_df_sum_agg.drop(columns=["time_period"])
la_emission_df_sum_agg.head()

,primary_id,emission_total
0,438438,3064.089389
1,438439,4969.142588
2,438440,4653.784493
3,438441,5253.589098
4,438442,5439.824263


### Emission data mean

In [68]:
# Filter out rows with time_period < 31
la_filtered_emission_total_df = la_emission_total_df[la_emission_total_df["time_period"] >= 31]
la_filtered_emission_total_df = la_filtered_emission_total_df.reset_index(drop=True)
la_filtered_emission_total_df.head(7)

,primary_id,time_period,emission_total
0,438438,31,14.093737
1,438438,32,7.732116
2,438438,33,1.889905
3,438438,34,-3.951764
4,438438,35,-10.104024
5,438439,31,145.247313
6,438439,32,143.261469


In [69]:
# aggregate data by primary_id by summing the emissions
la_emission_df_mean_agg = la_filtered_emission_total_df.groupby(["primary_id"]).mean().reset_index()

# Rename emission_total to emission_avg_last_five_years
la_emission_df_mean_agg.rename(columns={"emission_total": "emission_avg_last_five_years"}, inplace=True)
la_emission_df_mean_agg

,primary_id,time_period,emission_avg_last_five_years
0,438438,33.0,1.931994
1,438439,33.0,141.279585
2,438440,33.0,111.105432
3,438441,33.0,162.202210
4,438442,33.0,174.543479
...,...,...,...
973,439434,33.0,148.253160
974,439435,33.0,90.741423
975,439436,33.0,185.804305
976,439437,33.0,125.750665


In [70]:
# Drop year column as it is no longer needed
la_emission_df_mean_agg = la_emission_df_mean_agg.drop(columns=["time_period"])
la_emission_df_mean_agg.head()

,primary_id,emission_avg_last_five_years
0,438438,1.931994
1,438439,141.279585
2,438440,111.105432
3,438441,162.202210
4,438442,174.543479


### Combining emission agg into one df

In [71]:
print("la_emission_df_mean_agg shape:", la_emission_df_mean_agg.shape)
print("la_emission_df_sum_agg shape:", la_emission_df_sum_agg.shape)

la_emission_df_mean_agg shape: (978, 2)
la_emission_df_sum_agg shape: (978, 2)


In [72]:
la_emissions_df_merged = la_emission_df_mean_agg.merge(la_emission_df_sum_agg, on="primary_id", how="inner")
la_emissions_df_merged.head()

,primary_id,emission_avg_last_five_years,emission_total
0,438438,1.931994,3064.089389
1,438439,141.279585,4969.142588
2,438440,111.105432,4653.784493
3,438441,162.202210,5253.589098
4,438442,174.543479,5439.824263


In [73]:
print("la_emission_df_mean_agg shape:", la_emission_df_mean_agg.shape)

la_emission_df_mean_agg shape: (978, 2)


### Production and Production Cost Data

In [74]:
industry_cost_vars = [
    c for c in ind_fuel_demand_by_sector.columns
    if c.startswith("energy_demand_capex_") or c.startswith("energy_demand_opex_")
]

In [75]:
industry_product_vars =  [
    c for c in ind_fuel_demand_by_sector.columns
    if c.startswith("energy_demand_")
    and "_capex_" not in c
    and "_opex_" not in c
]

In [ ]:
ind_fuel_demand_by_sector[industry_cost_vars]

In [ ]:
ind_fuel_demand_by_sector[industry_product_vars]

#### production cost

In [76]:
# Sum all the subsector emission columns across axis=1
la_production_cost_total_df = ind_fuel_demand_by_sector.copy()
la_production_cost_total_df["production_cost_total"] = la_production_cost_total_df[industry_cost_vars].sum(axis=1)

# Show only the inputs that went into the sum + the result
la_production_cost_total_df[industry_cost_vars + ["production_cost_total"]].head()


,energy_demand_capex_agriculture_and_livestock_biomass,energy_demand_opex_agriculture_and_livestock_biomass,energy_demand_capex_chemicals_biomass,energy_demand_opex_chemicals_biomass,energy_demand_capex_electronics_biomass,energy_demand_opex_electronics_biomass,energy_demand_capex_glass_biomass,energy_demand_opex_glass_biomass,energy_demand_capex_mining_biomass,energy_demand_opex_mining_biomass,...,energy_demand_opex_mining_oil,energy_demand_capex_other_product_manufacturing_oil,energy_demand_opex_other_product_manufacturing_oil,energy_demand_capex_plastic_oil,energy_demand_opex_plastic_oil,energy_demand_capex_rubber_and_leather_oil,energy_demand_opex_rubber_and_leather_oil,energy_demand_capex_wood_oil,energy_demand_opex_wood_oil,production_cost_total
29,5.724716e+06,2.146769e+06,317965.809889,119237.178709,92.322045,34.620767,2513.271361,942.476760,6314.756484,2368.033681,...,49297.316368,51815.362954,19430.761108,779719.332924,292394.749847,124912.074038,46842.027764,1.930689e+06,724008.208766,7.563706e+08
30,5.372716e+06,2.014769e+06,298057.573990,111771.590246,105.837249,39.688968,2426.435685,909.913382,6601.088476,2475.408179,...,45096.616382,54769.061601,20538.398100,787120.223762,295170.083911,128433.029088,48162.385908,1.880348e+06,705130.527630,7.482375e+08
31,5.769727e+06,2.163648e+06,268656.188746,100746.070780,114.317824,42.869184,2362.109450,885.791044,6889.094315,2583.410368,...,40797.989216,57862.308602,21698.365726,797321.423344,298995.533754,132475.520039,49678.320015,1.815382e+06,680768.281295,7.553865e+08
32,5.742602e+06,2.153476e+06,238421.547821,89408.080433,122.032109,45.762041,2367.976822,887.991308,7158.142341,2684.303378,...,36388.660483,61298.374448,22986.890418,791895.271016,296960.726631,134238.504762,50339.439286,1.749552e+06,656082.065166,7.541000e+08
33,5.715628e+06,2.143361e+06,211588.495763,79345.685911,129.265101,48.474413,2462.619596,923.482349,7405.033048,2776.887393,...,31902.140999,64926.828759,24347.560784,767252.343702,287719.628888,133015.459640,49880.797365,1.689624e+06,633609.070712,7.532905e+08


In [77]:
# Keep only the primary_id, time_period, and emission_total columns
la_production_cost_total_df = la_production_cost_total_df[["primary_id", "time_period", "production_cost_total"]]
la_production_cost_total_df.head()

,primary_id,time_period,production_cost_total
29,438438,7,7.563706e+08
30,438438,8,7.482375e+08
31,438438,9,7.553865e+08
32,438438,10,7.541000e+08
33,438438,11,7.532905e+08


In [78]:
# aggregate data by primary_id summing the emissions
la_production_cost_df_mean_agg = la_production_cost_total_df.groupby(["primary_id"]).mean().reset_index()

# Drop time period column
la_production_cost_df_mean_agg = la_production_cost_df_mean_agg.drop(columns=["time_period"])
la_production_cost_df_mean_agg.head()

,primary_id,production_cost_total
0,438438,1.056679e+09
1,438439,1.095336e+09
2,438440,1.134537e+09
3,438441,1.098871e+09
4,438442,1.040931e+09


In [79]:
la_production_cost_df_mean_agg.shape

(978, 2)

#### production

In [80]:
# columns: base energy demand only (no capex/opex)
industry_product_vars = [
    c for c in ind_fuel_demand_by_sector.columns
    if c.startswith("energy_demand_") and "_capex_" not in c and "_opex_" not in c
]

la_production_total_df = ind_fuel_demand_by_sector.copy()
la_production_total_df["production_total"] = (
    la_production_total_df[industry_product_vars].sum(axis=1, min_count=1)
)

# show only what went into the sum + the result
la_production_total_df[industry_product_vars + ["production_total"]].head()


,energy_demand_enfu_subsector_total_pj_entc_fuel_biomass,energy_demand_agriculture_and_livestock_biomass,energy_demand_chemicals_biomass,energy_demand_electronics_biomass,energy_demand_glass_biomass,energy_demand_mining_biomass,energy_demand_other_product_manufacturing_biomass,energy_demand_paper_biomass,energy_demand_wood_biomass,energy_demand_enfu_subsector_total_pj_entc_fuel_coal,...,energy_demand_chemicals_oil,energy_demand_electronics_oil,energy_demand_glass_oil,energy_demand_mining_oil,energy_demand_other_product_manufacturing_oil,energy_demand_plastic_oil,energy_demand_rubber_and_leather_oil,energy_demand_wood_oil,energy_demand_enfu_subsector_total_pj_entc_fuel_crude,production_total
29,36.017886,5.148130,0.285941,0.000083,0.002260,0.005679,0.031915,67.212892,33.734585,63.003960,...,8.507847,0.005373,0.004605,0.118219,0.046597,0.701187,0.112331,1.736232,10719.051847,13003.410772
30,34.962556,4.831583,0.268038,0.000095,0.002182,0.005936,0.033135,65.984351,34.772445,37.816143,...,8.483177,0.005702,0.004902,0.108145,0.049253,0.707842,0.115497,1.690962,10500.513468,12771.316101
31,34.162696,5.188607,0.241597,0.000103,0.002124,0.006195,0.034061,64.830062,35.563369,68.398743,...,8.497075,0.005849,0.004732,0.097837,0.052034,0.717016,0.119133,1.632539,10332.462853,12606.960147
32,29.882798,5.164214,0.214408,0.000110,0.002129,0.006437,0.034908,63.317225,36.195323,51.019904,...,8.505441,0.006096,0.004524,0.087263,0.055124,0.712137,0.120718,1.573339,10352.089213,12629.790469
33,30.039144,5.139957,0.190278,0.000116,0.002215,0.006659,0.035757,61.510920,36.751135,51.099173,...,8.522053,0.006404,0.004484,0.076504,0.058387,0.689976,0.119618,1.519447,10323.027379,12603.257344


In [81]:
# Keep only the primary_id, time_period, and emission_total columns
la_production_total_df = la_production_total_df[["primary_id", "time_period", "production_total"]]
la_production_total_df.head()

,primary_id,time_period,production_total
29,438438,7,13003.410772
30,438438,8,12771.316101
31,438438,9,12606.960147
32,438438,10,12629.790469
33,438438,11,12603.257344


In [82]:
# aggregate data by primary_id summing the emissions
la_production_df_mean_agg = la_production_total_df.groupby(["primary_id"]).mean().reset_index()

# Drop time period column
la_production_df_mean_agg = la_production_df_mean_agg.drop(columns=["time_period"])
la_production_df_mean_agg.head()

,primary_id,production_total
0,438438,11079.466510
1,438439,12661.766840
2,438440,12705.369862
3,438441,13067.486466
4,438442,12974.920136


### CB data

In [83]:
# aggregate data by primary_id and region by summing the technical cost
cb_df_agg = cb_df.groupby(["primary_id", "future_id"]).mean().reset_index()
cb_df_agg

,primary_id,future_id,year,technical_cost,air_pollution
0,438438,0,2036.0,-11.910086,1.607136
1,438439,1,2036.0,-4.115464,0.940284
2,438440,2,2036.0,-6.955755,1.087341
3,438441,3,2036.0,-4.742205,1.034241
4,438442,4,2036.0,-4.583128,0.718387
...,...,...,...,...,...
973,439434,996,2036.0,-4.308308,1.137731
974,439435,997,2036.0,-7.153081,0.916891
975,439436,998,2036.0,-5.139749,0.472233
976,439437,999,2036.0,-13.847011,1.275897


In [84]:
# Drop year column as it is no longer needed
cb_df_agg = cb_df_agg.drop(columns=["year"], errors='ignore')
cb_df_agg.head()

,primary_id,future_id,technical_cost,air_pollution
0,438438,0,-11.910086,1.607136
1,438439,1,-4.115464,0.940284
2,438440,2,-6.955755,1.087341
3,438441,3,-4.742205,1.034241
4,438442,4,-4.583128,0.718387


In [85]:
cb_df_agg.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 978 entries, 0 to 977
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   primary_id      978 non-null    int64  
 1   future_id       978 non-null    int64  
 2   technical_cost  978 non-null    float64
 3   air_pollution   978 non-null    float64
dtypes: float64(2), int64(2)
memory usage: 30.7 KB


### LSU data

In [86]:
lsu_data

,primary_id,time,num_jobs,earning_per_job
1,438438,7,7.912604e-11,14971.586207
2,438438,8,-1.091394e-11,56661.333333
3,438438,9,4.547474e-11,491.52
4,438438,10,-1.637090e-11,56888.888889
5,438438,11,2.680509e+01,64709.62299
...,...,...,...,...
29335,439438,31,1.518724e+04,81026.402641
29336,439438,32,1.586526e+04,81151.92643
29337,439438,33,1.589483e+04,82113.052029
29338,439438,34,1.524502e+04,83941.264701


In [87]:
# aggregate data by primary_id and region by summing the technical cost
lsu_data_agg = lsu_data.groupby(["primary_id"]).mean().reset_index()
lsu_data_agg

,primary_id,time,num_jobs,earning_per_job
0,438438,21.0,22107.871075,58241.439771
1,438439,21.0,2518.479115,519262.615092
2,438440,21.0,1690.944493,66412.565922
3,438441,21.0,19821.037601,65756.314553
4,438442,21.0,908.045115,62500.939063
...,...,...,...,...
973,439434,21.0,14502.518083,-1122710.97297
974,439435,21.0,22088.655894,72885.582651
975,439436,21.0,-1060.925030,93097.048389
976,439437,21.0,12989.695809,113641.380315


## Merge emissions and cb data with lhs samples

In [88]:
#attr_primary_df = pd.read_csv(os.path.join(SIMULATION_DIR_PATH, "ATTRIBUTE_PRIMARY_6004_filtered_metamodel_version.csv"))
attr_primary_df

,primary_id,design_id,strategy_id,future_id
1,438438,4,6004,0
2,438439,4,6004,1
3,438440,4,6004,2
4,438441,4,6004,3
5,438442,4,6004,4
...,...,...,...,...
997,439434,4,6004,996
998,439435,4,6004,997
999,439436,4,6004,998
1000,439437,4,6004,999


In [89]:
# Check for duplicates in primary_id
duplicates_primary = attr_primary_df[attr_primary_df.duplicated(subset=["primary_id"], keep=False)]
if not duplicates_primary.empty:
    print("Duplicated primary_id found:")
    print(duplicates_primary)
else:
    print("No duplicated primary_id found.")

# Check for duplicates in future_id
duplicates_future = attr_primary_df[attr_primary_df.duplicated(subset=["future_id"], keep=False)]
if not duplicates_future.empty:
    print("Duplicated future_id found:")
    print(duplicates_future)
else:
    print("No duplicated future_id found.")


No duplicated primary_id found.
No duplicated future_id found.


In [90]:
la_emission_df_w_future_id = la_emissions_df_merged.merge(attr_primary_df, on="primary_id", how="inner")

# Drop design_id and stratgy_id columns
la_emission_df_w_future_id = la_emission_df_w_future_id.drop(columns=["design_id", "strategy_id"])
la_emission_df_w_future_id

,primary_id,emission_avg_last_five_years,emission_total,future_id
0,438438,1.931994,3064.089389,0
1,438439,141.279585,4969.142588,1
2,438440,111.105432,4653.784493,2
3,438441,162.202210,5253.589098,3
4,438442,174.543479,5439.824263,4
...,...,...,...,...
973,439434,148.253160,5101.470810,996
974,439435,90.741423,4314.619450,997
975,439436,185.804305,5548.147559,998
976,439437,125.750665,4838.484617,999


In [91]:
la_ssp_out_df = la_emission_df_w_future_id.merge(la_production_df_mean_agg, on="primary_id", how="inner")
la_ssp_out_df = la_ssp_out_df.merge(la_production_cost_df_mean_agg, on="primary_id", how="inner")
la_ssp_out_df.head()

,primary_id,emission_avg_last_five_years,emission_total,future_id,production_total,production_cost_total
0,438438,1.931994,3064.089389,0,11079.466510,1.056679e+09
1,438439,141.279585,4969.142588,1,12661.766840,1.095336e+09
2,438440,111.105432,4653.784493,2,12705.369862,1.134537e+09
3,438441,162.202210,5253.589098,3,13067.486466,1.098871e+09
4,438442,174.543479,5439.824263,4,12974.920136,1.040931e+09


In [92]:
# Check that the shape is correct
print("la_ssp_out_df shape:", la_ssp_out_df.shape)
print("la_emission_df_w_future_id shape:", la_emission_df_w_future_id.shape)
print("la_production_df_sum_agg shape:", la_production_df_mean_agg.shape)
print("la_production_cost_df_sum_agg shape:", la_production_cost_df_mean_agg.shape)

la_ssp_out_df shape: (978, 6)
la_emission_df_w_future_id shape: (978, 4)
la_production_df_sum_agg shape: (978, 2)
la_production_cost_df_sum_agg shape: (978, 2)


In [93]:
la_ssp_out_df.isna().sum()

primary_id                      0
emission_avg_last_five_years    0
emission_total                  0
future_id                       0
production_total                0
production_cost_total           0
dtype: int64

In [94]:
lhs_df_merged.head()

,future_id,46,47,48,49,50,51,52,53,54,...,1761,1762,1767,1768,1770,1778,1780,1792,1793,1801
1000,1,0.362295,0.639448,0.536505,0.069870,0.972121,0.350096,0.013272,0.138693,0.698405,...,0.440563,0.407390,0.239100,0.201661,0.933695,0.744614,0.345463,0.514843,0.040701,0.865033
1001,2,0.453799,0.702236,0.097084,0.604824,0.155687,0.670921,0.492146,0.863642,0.650856,...,0.186101,0.822272,0.409591,0.084974,0.769561,0.318526,0.942912,0.829314,0.229475,0.697621
1002,3,0.120669,0.576491,0.067843,0.362784,0.809979,0.282347,0.596430,0.791199,0.560186,...,0.542477,0.653018,0.966889,0.516578,0.754681,0.474827,0.817638,0.260621,0.728775,0.736069
1003,4,0.218794,0.895455,0.538409,0.193003,0.514389,0.493241,0.053878,0.546383,0.067101,...,0.906312,0.667512,0.420655,0.735931,0.249362,0.562946,0.791030,0.769653,0.512995,0.822182
1004,5,0.340851,0.766183,0.456184,0.561235,0.079006,0.883358,0.541472,0.215089,0.662216,...,0.166547,0.847204,0.379052,0.588185,0.687127,0.526062,0.511451,0.292011,0.825633,0.353985


In [95]:
lhs_emissions_merged_df = pd.merge(lhs_df_merged, la_ssp_out_df, on="future_id", how="inner")
lhs_emissions_merged_df.head()

,future_id,46,47,48,49,50,51,52,53,54,...,1778,1780,1792,1793,1801,primary_id,emission_avg_last_five_years,emission_total,production_total,production_cost_total
0,1,0.362295,0.639448,0.536505,0.069870,0.972121,0.350096,0.013272,0.138693,0.698405,...,0.744614,0.345463,0.514843,0.040701,0.865033,438439,141.279585,4969.142588,12661.766840,1.095336e+09
1,2,0.453799,0.702236,0.097084,0.604824,0.155687,0.670921,0.492146,0.863642,0.650856,...,0.318526,0.942912,0.829314,0.229475,0.697621,438440,111.105432,4653.784493,12705.369862,1.134537e+09
2,3,0.120669,0.576491,0.067843,0.362784,0.809979,0.282347,0.596430,0.791199,0.560186,...,0.474827,0.817638,0.260621,0.728775,0.736069,438441,162.202210,5253.589098,13067.486466,1.098871e+09
3,4,0.218794,0.895455,0.538409,0.193003,0.514389,0.493241,0.053878,0.546383,0.067101,...,0.562946,0.791030,0.769653,0.512995,0.822182,438442,174.543479,5439.824263,12974.920136,1.040931e+09
4,5,0.340851,0.766183,0.456184,0.561235,0.079006,0.883358,0.541472,0.215089,0.662216,...,0.526062,0.511451,0.292011,0.825633,0.353985,438443,91.608967,4363.325533,12229.952281,1.130174e+09


In [96]:
lhs_emissions_merged_df.shape

(977, 85)

In [97]:
cb_df_agg.head()

,primary_id,future_id,technical_cost,air_pollution
0,438438,0,-11.910086,1.607136
1,438439,1,-4.115464,0.940284
2,438440,2,-6.955755,1.087341
3,438441,3,-4.742205,1.034241
4,438442,4,-4.583128,0.718387


In [98]:
complete_merged_df = pd.merge(lhs_emissions_merged_df, cb_df_agg, on=["future_id", "primary_id"], how="inner")

complete_merged_df.head()

,future_id,46,47,48,49,50,51,52,53,54,...,1792,1793,1801,primary_id,emission_avg_last_five_years,emission_total,production_total,production_cost_total,technical_cost,air_pollution
0,1,0.362295,0.639448,0.536505,0.069870,0.972121,0.350096,0.013272,0.138693,0.698405,...,0.514843,0.040701,0.865033,438439,141.279585,4969.142588,12661.766840,1.095336e+09,-4.115464,0.940284
1,2,0.453799,0.702236,0.097084,0.604824,0.155687,0.670921,0.492146,0.863642,0.650856,...,0.829314,0.229475,0.697621,438440,111.105432,4653.784493,12705.369862,1.134537e+09,-6.955755,1.087341
2,3,0.120669,0.576491,0.067843,0.362784,0.809979,0.282347,0.596430,0.791199,0.560186,...,0.260621,0.728775,0.736069,438441,162.202210,5253.589098,13067.486466,1.098871e+09,-4.742205,1.034241
3,4,0.218794,0.895455,0.538409,0.193003,0.514389,0.493241,0.053878,0.546383,0.067101,...,0.769653,0.512995,0.822182,438442,174.543479,5439.824263,12974.920136,1.040931e+09,-4.583128,0.718387
4,5,0.340851,0.766183,0.456184,0.561235,0.079006,0.883358,0.541472,0.215089,0.662216,...,0.292011,0.825633,0.353985,438443,91.608967,4363.325533,12229.952281,1.130174e+09,-9.966251,1.143989


In [99]:
complete_merged_df

,future_id,46,47,48,49,50,51,52,53,54,...,1792,1793,1801,primary_id,emission_avg_last_five_years,emission_total,production_total,production_cost_total,technical_cost,air_pollution
0,1,0.362295,0.639448,0.536505,0.069870,0.972121,0.350096,0.013272,0.138693,0.698405,...,0.514843,0.040701,0.865033,438439,141.279585,4969.142588,12661.766840,1.095336e+09,-4.115464,0.940284
1,2,0.453799,0.702236,0.097084,0.604824,0.155687,0.670921,0.492146,0.863642,0.650856,...,0.829314,0.229475,0.697621,438440,111.105432,4653.784493,12705.369862,1.134537e+09,-6.955755,1.087341
2,3,0.120669,0.576491,0.067843,0.362784,0.809979,0.282347,0.596430,0.791199,0.560186,...,0.260621,0.728775,0.736069,438441,162.202210,5253.589098,13067.486466,1.098871e+09,-4.742205,1.034241
3,4,0.218794,0.895455,0.538409,0.193003,0.514389,0.493241,0.053878,0.546383,0.067101,...,0.769653,0.512995,0.822182,438442,174.543479,5439.824263,12974.920136,1.040931e+09,-4.583128,0.718387
4,5,0.340851,0.766183,0.456184,0.561235,0.079006,0.883358,0.541472,0.215089,0.662216,...,0.292011,0.825633,0.353985,438443,91.608967,4363.325533,12229.952281,1.130174e+09,-9.966251,1.143989
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
972,996,0.007594,0.203733,0.677713,0.495944,0.602643,0.012388,0.876806,0.114124,0.759058,...,0.082950,0.689295,0.339705,439434,148.253160,5101.470810,12131.810309,1.179379e+09,-4.308308,1.137731
973,997,0.086704,0.817248,0.202179,0.546244,0.617108,0.961640,0.575516,0.295376,0.902090,...,0.945989,0.890974,0.663466,439435,90.741423,4314.619450,11627.999061,1.167807e+09,-7.153081,0.916891
974,998,0.780116,0.485914,0.411919,0.088201,0.275818,0.610827,0.743325,0.409988,0.766523,...,0.939983,0.330520,0.799318,439436,185.804305,5548.147559,14393.657215,1.026944e+09,-5.139749,0.472233
975,999,0.903291,0.316209,0.375950,0.947067,0.724218,0.566487,0.563216,0.867955,0.201602,...,0.678928,0.001762,0.908362,439437,125.750665,4838.484617,14785.509642,1.156615e+09,-13.847011,1.275897


In [100]:
complete_merged_df = pd.merge(complete_merged_df, lsu_data_agg, on=[ "primary_id"], how="inner")
complete_merged_df

,future_id,46,47,48,49,50,51,52,53,54,...,primary_id,emission_avg_last_five_years,emission_total,production_total,production_cost_total,technical_cost,air_pollution,time,num_jobs,earning_per_job
0,1,0.362295,0.639448,0.536505,0.069870,0.972121,0.350096,0.013272,0.138693,0.698405,...,438439,141.279585,4969.142588,12661.766840,1.095336e+09,-4.115464,0.940284,21.0,2518.479115,519262.615092
1,2,0.453799,0.702236,0.097084,0.604824,0.155687,0.670921,0.492146,0.863642,0.650856,...,438440,111.105432,4653.784493,12705.369862,1.134537e+09,-6.955755,1.087341,21.0,1690.944493,66412.565922
2,3,0.120669,0.576491,0.067843,0.362784,0.809979,0.282347,0.596430,0.791199,0.560186,...,438441,162.202210,5253.589098,13067.486466,1.098871e+09,-4.742205,1.034241,21.0,19821.037601,65756.314553
3,4,0.218794,0.895455,0.538409,0.193003,0.514389,0.493241,0.053878,0.546383,0.067101,...,438442,174.543479,5439.824263,12974.920136,1.040931e+09,-4.583128,0.718387,21.0,908.045115,62500.939063
4,5,0.340851,0.766183,0.456184,0.561235,0.079006,0.883358,0.541472,0.215089,0.662216,...,438443,91.608967,4363.325533,12229.952281,1.130174e+09,-9.966251,1.143989,21.0,15529.498669,56867.313635
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
972,996,0.007594,0.203733,0.677713,0.495944,0.602643,0.012388,0.876806,0.114124,0.759058,...,439434,148.253160,5101.470810,12131.810309,1.179379e+09,-4.308308,1.137731,21.0,14502.518083,-1122710.97297
973,997,0.086704,0.817248,0.202179,0.546244,0.617108,0.961640,0.575516,0.295376,0.902090,...,439435,90.741423,4314.619450,11627.999061,1.167807e+09,-7.153081,0.916891,21.0,22088.655894,72885.582651
974,998,0.780116,0.485914,0.411919,0.088201,0.275818,0.610827,0.743325,0.409988,0.766523,...,439436,185.804305,5548.147559,14393.657215,1.026944e+09,-5.139749,0.472233,21.0,-1060.925030,93097.048389
975,999,0.903291,0.316209,0.375950,0.947067,0.724218,0.566487,0.563216,0.867955,0.201602,...,439437,125.750665,4838.484617,14785.509642,1.156615e+09,-13.847011,1.275897,21.0,12989.695809,113641.380315


In [101]:
print(complete_merged_df.shape)
print(complete_merged_df.future_id.nunique())

(977, 90)
977


In [102]:
complete_merged_df.isna().sum().sum()

np.int64(0)

In [103]:
# rearrange columns to have future_id and primary_id at the front
cols_order = ["future_id", "primary_id"] + [col for col in complete_merged_df.columns if col not in ["future_id", "primary_id"]]
complete_merged_df = complete_merged_df[cols_order]
complete_merged_df

,future_id,primary_id,46,47,48,49,50,51,52,53,...,1801,emission_avg_last_five_years,emission_total,production_total,production_cost_total,technical_cost,air_pollution,time,num_jobs,earning_per_job
0,1,438439,0.362295,0.639448,0.536505,0.069870,0.972121,0.350096,0.013272,0.138693,...,0.865033,141.279585,4969.142588,12661.766840,1.095336e+09,-4.115464,0.940284,21.0,2518.479115,519262.615092
1,2,438440,0.453799,0.702236,0.097084,0.604824,0.155687,0.670921,0.492146,0.863642,...,0.697621,111.105432,4653.784493,12705.369862,1.134537e+09,-6.955755,1.087341,21.0,1690.944493,66412.565922
2,3,438441,0.120669,0.576491,0.067843,0.362784,0.809979,0.282347,0.596430,0.791199,...,0.736069,162.202210,5253.589098,13067.486466,1.098871e+09,-4.742205,1.034241,21.0,19821.037601,65756.314553
3,4,438442,0.218794,0.895455,0.538409,0.193003,0.514389,0.493241,0.053878,0.546383,...,0.822182,174.543479,5439.824263,12974.920136,1.040931e+09,-4.583128,0.718387,21.0,908.045115,62500.939063
4,5,438443,0.340851,0.766183,0.456184,0.561235,0.079006,0.883358,0.541472,0.215089,...,0.353985,91.608967,4363.325533,12229.952281,1.130174e+09,-9.966251,1.143989,21.0,15529.498669,56867.313635
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
972,996,439434,0.007594,0.203733,0.677713,0.495944,0.602643,0.012388,0.876806,0.114124,...,0.339705,148.253160,5101.470810,12131.810309,1.179379e+09,-4.308308,1.137731,21.0,14502.518083,-1122710.97297
973,997,439435,0.086704,0.817248,0.202179,0.546244,0.617108,0.961640,0.575516,0.295376,...,0.663466,90.741423,4314.619450,11627.999061,1.167807e+09,-7.153081,0.916891,21.0,22088.655894,72885.582651
974,998,439436,0.780116,0.485914,0.411919,0.088201,0.275818,0.610827,0.743325,0.409988,...,0.799318,185.804305,5548.147559,14393.657215,1.026944e+09,-5.139749,0.472233,21.0,-1060.925030,93097.048389
975,999,439437,0.903291,0.316209,0.375950,0.947067,0.724218,0.566487,0.563216,0.867955,...,0.908362,125.750665,4838.484617,14785.509642,1.156615e+09,-13.847011,1.275897,21.0,12989.695809,113641.380315


In [104]:
complete_merged_df = complete_merged_df.drop(columns=["time"])

In [105]:
# Check for nans
complete_merged_df.isna().sum().sum()

np.int64(0)

## Filter out irrelevant lhs groups

In [106]:
var_traj_X_df = pd.read_csv(os.path.join(SIMULATION_DIR_PATH, "VARIABLE_TRAJECTORY_GROUPS_X.csv"))
var_traj_L_df = pd.read_csv(os.path.join(SIMULATION_DIR_PATH, "VARIABLE_TRAJECTORY_GROUPS_L.csv"))

In [107]:
var_traj_X_df.tail()

,variable,variable_trajectory_group
79,nemomod_entc_variable_cost_pp_solar_usd_per_mwh,57
80,nemomod_entc_variable_cost_pp_wind_usd_per_mwh,57
81,elasticity_trde_mtkm_to_gdp_freight,58
82,elasticity_trde_pkm_to_gdppc_private_and_public,59
83,elasticity_trde_pkm_to_gdppc_regional,59


In [108]:
var_traj_L_df.tail()

,transformation_code,variable,variable_field,variable_trajectory_group
456,TX:WASO:INC_RECYCLING,Fraction of Waste Recycled,frac_waso_recycled_paper,45
457,TX:WASO:INC_RECYCLING,Fraction of Waste Recycled,frac_waso_recycled_plastic,45
458,TX:WASO:INC_RECYCLING,Fraction of Waste Recycled,frac_waso_recycled_rubber_leather,45
459,TX:WASO:INC_RECYCLING,Fraction of Waste Recycled,frac_waso_recycled_textiles,45
460,TX:WASO:INC_RECYCLING,Fraction of Waste Recycled,frac_waso_recycled_wood,45


In [109]:
var_traj_groups_X = var_traj_X_df["variable_trajectory_group"].unique()
var_traj_groups_L = var_traj_L_df["variable_trajectory_group"].unique()
print("Variable trajectory groups X:", var_traj_groups_X)
print("Variable trajectory groups L:", var_traj_groups_L)

Variable trajectory groups X: [46 47 48 49 50 51 52 53 54 55 56 57 58 59]
Variable trajectory groups L: [ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24
 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45]


In [110]:
# join the var_traj_groups
var_traj_groups_all = var_traj_groups_X.tolist() + var_traj_groups_L.tolist()
var_traj_groups_all = list(set(var_traj_groups_all))  # remove duplicates
print("All variable trajectory groups:", var_traj_groups_all)

All variable trajectory groups: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59]


In [111]:
# Convert the variable_trajectory_group column to list of strings
relevant_lhs_cols = [str(col) for col in var_traj_groups_all]

In [112]:
df_cols = complete_merged_df.columns.tolist()

# Filter the relevant_lhs_cols to only include those that are in df_cols
relevant_lhs_cols = [col for col in relevant_lhs_cols if col in df_cols]

In [113]:
# filter complete_merged_df to keep only relevant columns
cols_to_keep = ["future_id", "primary_id"] + list(relevant_lhs_cols) + ["emission_avg_last_five_years", "emission_total", "production_total","production_cost_total","technical_cost", "air_pollution", "num_jobs","earning_per_job"]
merged_df_filtered = complete_merged_df[cols_to_keep]

In [114]:
print("Original merged DataFrame shape:", complete_merged_df.shape)
print("Filtered merged DataFrame shape:", merged_df_filtered.shape)

Original merged DataFrame shape: (977, 89)
Filtered merged DataFrame shape: (977, 69)


In [115]:
print("Filtered merged DataFrame fields:", merged_df_filtered.columns.tolist())
print("Relevant LHS columns:", relevant_lhs_cols)

Filtered merged DataFrame fields: ['future_id', 'primary_id', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50', '51', '52', '53', '54', '55', '56', '57', '58', '59', 'emission_avg_last_five_years', 'emission_total', 'production_total', 'production_cost_total', 'technical_cost', 'air_pollution', 'num_jobs', 'earning_per_job']
Relevant LHS columns: ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50', '51', '52', '53', '54', '55', '56', '57', '58', '59']


In [116]:
merged_df_filtered.head()

,future_id,primary_id,1,2,3,4,5,6,7,8,...,58,59,emission_avg_last_five_years,emission_total,production_total,production_cost_total,technical_cost,air_pollution,num_jobs,earning_per_job
0,1,438439,0.303285,0.037934,0.617066,0.097521,0.469836,0.216534,0.944742,0.040950,...,0.323321,0.466898,141.279585,4969.142588,12661.766840,1.095336e+09,-4.115464,0.940284,2518.479115,519262.615092
1,2,438440,0.377135,0.539486,0.857265,0.025950,0.012477,0.783614,0.586953,0.735174,...,0.229789,0.188339,111.105432,4653.784493,12705.369862,1.134537e+09,-6.955755,1.087341,1690.944493,66412.565922
2,3,438441,0.952401,0.228593,0.972167,0.960658,0.159089,0.039879,0.327936,0.331580,...,0.195396,0.332572,162.202210,5253.589098,13067.486466,1.098871e+09,-4.742205,1.034241,19821.037601,65756.314553
3,4,438442,0.765244,0.871007,0.325361,0.045184,0.537098,0.040606,0.761664,0.575896,...,0.664778,0.138639,174.543479,5439.824263,12974.920136,1.040931e+09,-4.583128,0.718387,908.045115,62500.939063
4,5,438443,0.082978,0.171474,0.720600,0.619098,0.797104,0.821825,0.820461,0.125243,...,0.417463,0.635064,91.608967,4363.325533,12229.952281,1.130174e+09,-9.966251,1.143989,15529.498669,56867.313635


## Add variable names to lhs columns

In [117]:
var_traj_L_df.tail()

,transformation_code,variable,variable_field,variable_trajectory_group
456,TX:WASO:INC_RECYCLING,Fraction of Waste Recycled,frac_waso_recycled_paper,45
457,TX:WASO:INC_RECYCLING,Fraction of Waste Recycled,frac_waso_recycled_plastic,45
458,TX:WASO:INC_RECYCLING,Fraction of Waste Recycled,frac_waso_recycled_rubber_leather,45
459,TX:WASO:INC_RECYCLING,Fraction of Waste Recycled,frac_waso_recycled_textiles,45
460,TX:WASO:INC_RECYCLING,Fraction of Waste Recycled,frac_waso_recycled_wood,45


In [118]:
var_traj_L_df = var_traj_L_df[["variable_field", "variable_trajectory_group"]]
var_traj_L_df = var_traj_L_df.rename(columns={"variable_field": "variable"})
var_traj_L_df.head()

,variable,variable_trajectory_group
0,ef_agrc_anaerobicdom_rice_kg_ch4_ha,1
1,frac_agrc_agriculture_production_lost,2
2,frac_agrc_crop_residues_burned,3
3,frac_agrc_crop_residues_removed,3
4,frac_agrc_no_till_cereals,3


In [119]:
var_traj_X_df.tail()

,variable,variable_trajectory_group
79,nemomod_entc_variable_cost_pp_solar_usd_per_mwh,57
80,nemomod_entc_variable_cost_pp_wind_usd_per_mwh,57
81,elasticity_trde_mtkm_to_gdp_freight,58
82,elasticity_trde_pkm_to_gdppc_private_and_public,59
83,elasticity_trde_pkm_to_gdppc_regional,59


In [120]:
var_traj_all_df = pd.concat([var_traj_X_df, var_traj_L_df], ignore_index=True)
var_traj_all_df

,variable,variable_trajectory_group
0,exports_enfu_pj_fuel_ammonia,46
1,exports_enfu_pj_fuel_coal,46
2,exports_enfu_pj_fuel_crude,46
3,exports_enfu_pj_fuel_diesel,46
4,exports_enfu_pj_fuel_electricity,46
...,...,...
540,frac_waso_recycled_paper,45
541,frac_waso_recycled_plastic,45
542,frac_waso_recycled_rubber_leather,45
543,frac_waso_recycled_textiles,45


In [121]:
# drop duplicates if any
print("Before dropping duplicates, var_traj_all_df shape:", var_traj_all_df.shape)
var_traj_all_df = var_traj_all_df.drop_duplicates(subset=["variable", "variable_trajectory_group"])
print("After dropping duplicates, var_traj_all_df shape:", var_traj_all_df.shape)

Before dropping duplicates, var_traj_all_df shape: (545, 2)
After dropping duplicates, var_traj_all_df shape: (537, 2)


In [122]:
# check if there are any duplicated variable names
duplicated_vars = var_traj_all_df["variable"].duplicated().any()
if duplicated_vars:
    print("There are duplicated variable names in var_traj_all_df.")
else:
    print("No duplicated variable names in var_traj_all_df.")

No duplicated variable names in var_traj_all_df.


In [123]:
# Filter var_traj_all_df by sample_group in relevant_lhs_cols
relevant_lhs_cols = [int(col) for col in relevant_lhs_cols]
var_traj_all_df = var_traj_all_df[var_traj_all_df["variable_trajectory_group"].isin(relevant_lhs_cols)]
var_traj_all_df = var_traj_all_df.sort_values(by="variable_trajectory_group", ascending=True)
print("After filtering by relevant_lhs_cols, var_traj_all_df shape:", var_traj_all_df.shape)

After filtering by relevant_lhs_cols, var_traj_all_df shape: (537, 2)


In [124]:
def process_variable_prefix(df):
    result = []
    for group, group_df in df.groupby('variable_trajectory_group'):
        variables = group_df['variable'].tolist()
        if len(variables) == 1:
            prefix = variables[0]
        else:
            prefix = os.path.commonprefix(variables)
            # Clean trailing underscores
            prefix = prefix.rstrip('_')
            
        prefix = f"group_{group}_{prefix}"
        result.append({'variable_trajectory_group': group, 'variable_prefix': prefix})
    return pd.DataFrame(result)

prefix_df = process_variable_prefix(var_traj_all_df)
prefix_df

,variable_trajectory_group,variable_prefix
0,1,group_1_ef_agrc_anaerobicdom_rice_kg_ch4_ha
1,2,group_2_frac_agrc_agriculture_production_lost
2,3,group_3_frac_agrc
3,4,group_4_qty_ccsq_mt_co2_captured_sequestered_b...
4,5,group_5_nemomod_entc_frac_min_share_production...
5,6,group_6_nemomod_en
6,7,group_7_
7,8,group_8_frac_fgtv_drained_and_waste_ch4_flared...
8,9,group_9_efficfactor_enfu_industrial_energy_fuel
9,10,group_10_scalar_inen_energy_demand


In [125]:
# Check for duplicates in variable_trajectory_group and variable_prefix
dups = prefix_df.duplicated(subset=["variable_trajectory_group", "variable_prefix"], keep=False)
if dups.any():
    print("Duplicated variable_trajectory_group and variable_prefix found:")
    print(prefix_df[dups])
else:
    print("No duplicated variable_trajectory_group and variable_prefix found.")

# Check for duplicates in variable_trajectory_group
dups_group = prefix_df.duplicated(subset=["variable_trajectory_group"], keep=False)
if dups_group.any():
    print("Duplicated variable_trajectory_group found:")
    print(prefix_df[dups_group])
else:
    print("No duplicated variable_trajectory_group found.")

# Check for duplicates in variable_prefix
dups_prefix = prefix_df.duplicated(subset=["variable_prefix"], keep=False)
if dups_prefix.any():
    print("Duplicated variable_prefix found:")
    print(prefix_df[dups_prefix])
else:
    print("No duplicated variable_prefix found.")

No duplicated variable_trajectory_group and variable_prefix found.
No duplicated variable_trajectory_group found.
No duplicated variable_prefix found.


In [ ]:
# var_traj_all_df[var_traj_all_df["variable_trajectory_group"].isin([3, 13, 40])]

In [ ]:
# prefix_df.loc[prefix_df["sample_group"] == 13, "variable_prefix"] = "group_13_frac_gnrl_eating_red_meats+"
# prefix_df.loc[prefix_df["sample_group"] == 40, "variable_prefix"] = "group_40_pij_lndu_grasslands+"

# prefix_df = prefix_df.sort_values(by="variable_prefix", ascending=True)
# prefix_df

In [126]:
# Let's use the prefix_df to rename the columns in merged_df_filtered
def rename_columns_with_prefix(merged_df, prefix_df):
    df = merged_df.copy()
    # Create a mapping from str(group) to prefix
    group_to_prefix = {str(row['variable_trajectory_group']): row['variable_prefix'] for _, row in prefix_df.iterrows()}
    # Only rename columns that match a group
    rename_dict = {col: group_to_prefix[col] for col in df.columns if col in group_to_prefix}
    df = df.rename(columns=rename_dict)
    return df

merged_df_filtered_w_prefix = rename_columns_with_prefix(merged_df_filtered, prefix_df)

In [127]:
merged_df_filtered_w_prefix

,future_id,primary_id,group_1_ef_agrc_anaerobicdom_rice_kg_ch4_ha,group_2_frac_agrc_agriculture_production_lost,group_3_frac_agrc,group_4_qty_ccsq_mt_co2_captured_sequestered_by_direct_air_capture,group_5_nemomod_entc_frac_min_share_production_fp_hydrogen,group_6_nemomod_en,group_7_,group_8_frac_fgtv_drained_and_waste_ch4_flared_fuel,...,group_58_elasticity_trde_mtkm_to_gdp_freight,group_59_elasticity_trde_pkm_to_gdppc,emission_avg_last_five_years,emission_total,production_total,production_cost_total,technical_cost,air_pollution,num_jobs,earning_per_job
0,1,438439,0.303285,0.037934,0.617066,0.097521,0.469836,0.216534,0.944742,0.040950,...,0.323321,0.466898,141.279585,4969.142588,12661.766840,1.095336e+09,-4.115464,0.940284,2518.479115,519262.615092
1,2,438440,0.377135,0.539486,0.857265,0.025950,0.012477,0.783614,0.586953,0.735174,...,0.229789,0.188339,111.105432,4653.784493,12705.369862,1.134537e+09,-6.955755,1.087341,1690.944493,66412.565922
2,3,438441,0.952401,0.228593,0.972167,0.960658,0.159089,0.039879,0.327936,0.331580,...,0.195396,0.332572,162.202210,5253.589098,13067.486466,1.098871e+09,-4.742205,1.034241,19821.037601,65756.314553
3,4,438442,0.765244,0.871007,0.325361,0.045184,0.537098,0.040606,0.761664,0.575896,...,0.664778,0.138639,174.543479,5439.824263,12974.920136,1.040931e+09,-4.583128,0.718387,908.045115,62500.939063
4,5,438443,0.082978,0.171474,0.720600,0.619098,0.797104,0.821825,0.820461,0.125243,...,0.417463,0.635064,91.608967,4363.325533,12229.952281,1.130174e+09,-9.966251,1.143989,15529.498669,56867.313635
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
972,996,439434,0.472939,0.189456,0.907254,0.382720,0.347644,0.188415,0.076790,0.158025,...,0.989099,0.514156,148.253160,5101.470810,12131.810309,1.179379e+09,-4.308308,1.137731,14502.518083,-1122710.97297
973,997,439435,0.193303,0.530631,0.754175,0.982186,0.440327,0.548382,0.257368,0.987097,...,0.648940,0.678492,90.741423,4314.619450,11627.999061,1.167807e+09,-7.153081,0.916891,22088.655894,72885.582651
974,998,439436,0.676414,0.041240,0.393548,0.099607,0.403658,0.391504,0.296141,0.005899,...,0.487303,0.981028,185.804305,5548.147559,14393.657215,1.026944e+09,-5.139749,0.472233,-1060.925030,93097.048389
975,999,439437,0.462831,0.540626,0.381429,0.621084,0.336953,0.649814,0.021256,0.553394,...,0.833030,0.285207,125.750665,4838.484617,14785.509642,1.156615e+09,-13.847011,1.275897,12989.695809,113641.380315


In [128]:
print(merged_df_filtered.shape)
print(merged_df_filtered_w_prefix.shape)

(977, 69)
(977, 69)


In [129]:
# check for duplicated column names
duplicated_cols = merged_df_filtered_w_prefix.columns[merged_df_filtered_w_prefix.columns.duplicated()].tolist()
if duplicated_cols:
    print("Duplicated column names found:", duplicated_cols)
else:
    print("No duplicated column names found.")

No duplicated column names found.


## Finally we save the processed data as training data

In [130]:
#save the merged DataFrame to a CSV file
merged_df_filtered_w_prefix.to_csv(os.path.join(TRAINING_DIR_PATH, "training_data_v5.9.csv"), index=False)

In [131]:
merged_df_filtered_w_prefix

,future_id,primary_id,group_1_ef_agrc_anaerobicdom_rice_kg_ch4_ha,group_2_frac_agrc_agriculture_production_lost,group_3_frac_agrc,group_4_qty_ccsq_mt_co2_captured_sequestered_by_direct_air_capture,group_5_nemomod_entc_frac_min_share_production_fp_hydrogen,group_6_nemomod_en,group_7_,group_8_frac_fgtv_drained_and_waste_ch4_flared_fuel,...,group_58_elasticity_trde_mtkm_to_gdp_freight,group_59_elasticity_trde_pkm_to_gdppc,emission_avg_last_five_years,emission_total,production_total,production_cost_total,technical_cost,air_pollution,num_jobs,earning_per_job
0,1,438439,0.303285,0.037934,0.617066,0.097521,0.469836,0.216534,0.944742,0.040950,...,0.323321,0.466898,141.279585,4969.142588,12661.766840,1.095336e+09,-4.115464,0.940284,2518.479115,519262.615092
1,2,438440,0.377135,0.539486,0.857265,0.025950,0.012477,0.783614,0.586953,0.735174,...,0.229789,0.188339,111.105432,4653.784493,12705.369862,1.134537e+09,-6.955755,1.087341,1690.944493,66412.565922
2,3,438441,0.952401,0.228593,0.972167,0.960658,0.159089,0.039879,0.327936,0.331580,...,0.195396,0.332572,162.202210,5253.589098,13067.486466,1.098871e+09,-4.742205,1.034241,19821.037601,65756.314553
3,4,438442,0.765244,0.871007,0.325361,0.045184,0.537098,0.040606,0.761664,0.575896,...,0.664778,0.138639,174.543479,5439.824263,12974.920136,1.040931e+09,-4.583128,0.718387,908.045115,62500.939063
4,5,438443,0.082978,0.171474,0.720600,0.619098,0.797104,0.821825,0.820461,0.125243,...,0.417463,0.635064,91.608967,4363.325533,12229.952281,1.130174e+09,-9.966251,1.143989,15529.498669,56867.313635
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
972,996,439434,0.472939,0.189456,0.907254,0.382720,0.347644,0.188415,0.076790,0.158025,...,0.989099,0.514156,148.253160,5101.470810,12131.810309,1.179379e+09,-4.308308,1.137731,14502.518083,-1122710.97297
973,997,439435,0.193303,0.530631,0.754175,0.982186,0.440327,0.548382,0.257368,0.987097,...,0.648940,0.678492,90.741423,4314.619450,11627.999061,1.167807e+09,-7.153081,0.916891,22088.655894,72885.582651
974,998,439436,0.676414,0.041240,0.393548,0.099607,0.403658,0.391504,0.296141,0.005899,...,0.487303,0.981028,185.804305,5548.147559,14393.657215,1.026944e+09,-5.139749,0.472233,-1060.925030,93097.048389
975,999,439437,0.462831,0.540626,0.381429,0.621084,0.336953,0.649814,0.021256,0.553394,...,0.833030,0.285207,125.750665,4838.484617,14785.509642,1.156615e+09,-13.847011,1.275897,12989.695809,113641.380315
